# 04 - Treat Admissions (Home Care)

## Input
- `data/raw/admission/` — homecare files + combined files 2019–2024

## Output
- `data/clean/homecare_admissions_by_acpr.csv` — one row per ACPR x year

## Columns
- `year`, `state`, `acpr_code`, `acpr_name`
- `n_first_admission`, `n_repeat_admission`
- `hcp_l1`, `hcp_l2`, `hcp_l3`, `hcp_l4`
- `n_age_<group>`
- `n_male`, `n_female`
- `n_indigenous`
- `n_nesb`, `n_english_speaking`

## Note
- Each row in raw data = one admission event
- Keeps Home care only, drops all other care types

In [33]:
import pandas as pd
import numpy as np
import os
import re

RAW_ADM  = '../../data/raw/admission'
OUT_HOME = '../../data/clean/homecare_admissions_by_acpr.csv'

WANTED = {
    'ACPR_code', 'ACPR_CODE', 'ACPR_CODE_2018',
    'ACPR_name', 'ACPR_NAME', 'ACPR_NAME_2018',
    'State', 'STATE',
    'Year', 'YEAR', 'Financial_year', 'FINANCIAL_YEAR',
    'Care_type', 'CARE_TYPE',
    'First_admission', 'First _admission', 'FIRST_ADMISSION',
    'Home_care_level', 'HOME_CARE_LEVEL',
    'Age_group', 'AGE_GROUP', 'AGE_GROUP_5',
    'Sex', 'SEX',
    'Indigenous_status', 'INDIGENOUS_STATUS',
    'Country_of_birth', 'COUNTRY_OF_BIRTH',
}

COL_MAP = {
    'ACPR_code': 'acpr_code', 'ACPR_CODE': 'acpr_code', 'ACPR_CODE_2018': 'acpr_code',
    'ACPR_name': 'acpr_name', 'ACPR_NAME': 'acpr_name', 'ACPR_NAME_2018': 'acpr_name',
    'State': 'state', 'STATE': 'state',
    'Year': 'year', 'YEAR': 'year', 'Financial_year': 'year', 'FINANCIAL_YEAR': 'year',
    'Care_type': 'care_type', 'CARE_TYPE': 'care_type',
    'First_admission': 'first_admission', 'First _admission': 'first_admission', 'FIRST_ADMISSION': 'first_admission',
    'Home_care_level': 'hcp_level', 'HOME_CARE_LEVEL': 'hcp_level',
    'Age_group': 'age_group', 'AGE_GROUP': 'age_group', 'AGE_GROUP_5': 'age_group',
    'Sex': 'sex', 'SEX': 'sex',
    'Indigenous_status': 'indigenous_status', 'INDIGENOUS_STATUS': 'indigenous_status',
    'Country_of_birth': 'country_of_birth', 'COUNTRY_OF_BIRTH': 'country_of_birth',
}

GROUP_COLS = ['acpr_code', 'acpr_name', 'state', 'year']

# Age bracket lookup: hyphen format, en-dash format, 4-digit format
AGE_MAP = {
    '0-49': 'n_age_0_49',   '0–49': 'n_age_0_49',   '0049': 'n_age_0_49',
    '50-54': 'n_age_50_54', '50–54': 'n_age_50_54', '5054': 'n_age_50_54',
    '55-59': 'n_age_55_59', '55–59': 'n_age_55_59', '5559': 'n_age_55_59',
    '60-64': 'n_age_60_64', '60–64': 'n_age_60_64', '6064': 'n_age_60_64',
    '65-69': 'n_age_65_69', '65–69': 'n_age_65_69', '6569': 'n_age_65_69',
    '70-74': 'n_age_70_74', '70–74': 'n_age_70_74', '7074': 'n_age_70_74',
    '75-79': 'n_age_75_79', '75–79': 'n_age_75_79', '7579': 'n_age_75_79',
    '80-84': 'n_age_80_84', '80–84': 'n_age_80_84', '8084': 'n_age_80_84',
    '85-89': 'n_age_85_89', '85–89': 'n_age_85_89', '8589': 'n_age_85_89',
    '90-94': 'n_age_90_94', '90–94': 'n_age_90_94', '9094': 'n_age_90_94',
    '95-99': 'n_age_95_99', '95–99': 'n_age_95_99', '9599': 'n_age_95_99',
    '100+': 'n_age_100_plus',
}
AGE_COLS = [
    'n_age_0_49', 'n_age_50_54', 'n_age_55_59', 'n_age_60_64',
    'n_age_65_69', 'n_age_70_74', 'n_age_75_79', 'n_age_80_84',
    'n_age_85_89', 'n_age_90_94', 'n_age_95_99', 'n_age_100_plus',
]

HCP_MAP = {
    '1': 'hcp_l1', 'l1': 'hcp_l1', 'level 1': 'hcp_l1', 'level1': 'hcp_l1',
    '2': 'hcp_l2', 'l2': 'hcp_l2', 'level 2': 'hcp_l2', 'level2': 'hcp_l2',
    '3': 'hcp_l3', 'l3': 'hcp_l3', 'level 3': 'hcp_l3', 'level3': 'hcp_l3',
    '4': 'hcp_l4', 'l4': 'hcp_l4', 'level 4': 'hcp_l4', 'level4': 'hcp_l4',
}
HCP_COLS = ['hcp_l1', 'hcp_l2', 'hcp_l3', 'hcp_l4']

In [34]:
# =============================================================================
# STEP 1 — Helper functions
# =============================================================================

def load_file(path):
    if path.endswith('.xlsx'):
        try:
            return pd.read_excel(path, engine='calamine')
        except Exception:
            return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1', low_memory=False)


def resolve_year(fname):
    # Use the END year of the financial year from the filename
    # e.g. '2019-20' or '2019–20' → 2020, '2023-24' → 2024
    # Handles both hyphen (-) and en-dash (–) in filenames
    m = re.search(r'20\d{2}[-–](\d{2})', fname)
    return int('20' + m.group(1)) if m else np.nan


def aggregate_file(df, fname):
    # 1. Slim and rename
    keep = [c for c in df.columns if c in WANTED]
    df = df[keep].copy()
    df = df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})

    if 'acpr_code' not in df.columns:
        print(f'  SKIP {fname} — no acpr_code')
        return None

    # 2. Assign year from filename, drop bad rows
    df['year'] = resolve_year(fname)
    df['acpr_code'] = df['acpr_code'].astype(str).str.strip()
    df = df[df['acpr_code'].str.len() > 0]
    df = df.dropna(subset=['acpr_code'])
    if df.empty:
        print(f'  SKIP {fname} — empty after dropna')
        return None
    df['year'] = int(df['year'].iloc[0])

    # 3. Keep Home care only
    if 'care_type' in df.columns:
        df = df[df['care_type'].astype(str).str.strip().str.lower() == 'home care']
    if df.empty:
        print(f'  SKIP {fname} — no home care rows')
        return None

    gcols = [c for c in GROUP_COLS if c in df.columns]

    # 4. Binary flag columns per row
    fa = df['first_admission'].astype(str).str.strip().str.lower() if 'first_admission' in df.columns else pd.Series('', index=df.index)
    df['_first']  = (fa == 'yes').astype(int)
    df['_repeat'] = (fa == 'no').astype(int)

    ind = df['indigenous_status'].astype(str).str.strip().str.lower() if 'indigenous_status' in df.columns else pd.Series('', index=df.index)
    df['_indigenous'] = (ind == 'indigenous').astype(int)

    sex = df['sex'].astype(str).str.strip().str.lower() if 'sex' in df.columns else pd.Series('', index=df.index)
    df['_male']   = sex.isin(['male', 'm', '1']).astype(int)
    df['_female'] = sex.isin(['female', 'f', '2']).astype(int)

    cob = df['country_of_birth'].astype(str).str.strip().str.lower() if 'country_of_birth' in df.columns else pd.Series('', index=df.index)
    df['_nesb']    = cob.str.contains('non-english', na=False).astype(int)
    df['_english'] = (cob.str.contains('english', na=False) & ~cob.str.contains('non-english', na=False)).astype(int)

    # 5. HCP level flags
    if 'hcp_level' in df.columns:
        hcp_norm = df['hcp_level'].astype(str).str.strip().str.lower().map(HCP_MAP)
        for col in HCP_COLS:
            df[f'_{col}'] = (hcp_norm == col).astype(int)
    else:
        for col in HCP_COLS:
            df[f'_{col}'] = 0

    # 6. Age bracket flags
    if 'age_group' in df.columns:
        age_norm = df['age_group'].astype(str).str.strip().map(AGE_MAP)
        for col in AGE_COLS:
            df[f'_{col}'] = (age_norm == col).astype(int)
    else:
        for col in AGE_COLS:
            df[f'_{col}'] = 0

    # 7. Aggregate: sum all flags per ACPR x year
    agg_map = {
        'n_first_admission':  ('_first',      'sum'),
        'n_repeat_admission': ('_repeat',     'sum'),
        'n_indigenous':       ('_indigenous', 'sum'),
        'n_male':             ('_male',       'sum'),
        'n_female':           ('_female',     'sum'),
        'n_nesb':             ('_nesb',       'sum'),
        'n_english_speaking': ('_english',    'sum'),
    }
    for col in HCP_COLS:
        agg_map[col] = (f'_{col}', 'sum')
    for col in AGE_COLS:
        agg_map[col] = (f'_{col}', 'sum')

    return df.groupby(gcols).agg(**agg_map).reset_index()


# =============================================================================
# STEP 2 — Main loop
# =============================================================================

results = []
for fname in sorted(os.listdir(RAW_ADM)):
    if fname.startswith('~'): continue
    if not (fname.endswith('.xlsx') or fname.endswith('.csv')): continue
    if 'residential' in fname.lower(): continue

    path = f'{RAW_ADM}/{fname}'
    try:
        df  = load_file(path)
        agg = aggregate_file(df, fname)
        if agg is not None:
            results.append(agg)
            print(f'OK  {fname}: {df.shape} → {agg.shape}  year={agg["year"].iloc[0]}')
        del df
    except Exception as e:
        print(f'ERR {fname}: {e}')

summary = pd.concat(results, ignore_index=True)
print(f'\nCombined: {summary.shape}')
print('Years:', sorted(summary['year'].unique()))
print('ACPRs:', summary['acpr_code'].nunique())

OK  Admissions-into-aged-care-2022-23.csv: (316558, 12) → (71, 27)  year=2023
OK  Admissionshomecare_2019–20_GENdata.xlsx: (94605, 12) → (85, 27)  year=2020
OK  Admissionshomecare_2020–21_GENdata.xlsx: (100436, 12) → (73, 26)  year=2021
OK  Admissionshomecare_2021-22_GENdata.csv: (95157, 12) → (154, 27)  year=2022
OK  CURF_Admissions-into-aged-care_2023-24.xlsx: (298953, 12) → (71, 27)  year=2024

Combined: (454, 27)
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
ACPRs: 74


In [35]:
# =============================================================================
# STEP 3 — Quick sanity check
# =============================================================================

print(summary[['acpr_name', 'year',
               'n_first_admission', 'n_repeat_admission',
               'n_indigenous', 'n_nesb',
               'n_male', 'n_female',
               'hcp_l1', 'hcp_l2', 'hcp_l3', 'hcp_l4']].head(10).to_string(index=False))
print()
print('Age cols sample:')
print(summary[['acpr_name', 'year'] + AGE_COLS].head(5).to_string(index=False))

      acpr_name  year  n_first_admission  n_repeat_admission  n_indigenous  n_nesb  n_male  n_female  hcp_l1  hcp_l2  hcp_l3  hcp_l4
  Central Coast  2023               1911                 588            49     220     992      1507     211    1198     824     266
   Central West  2023               3463                 219           380     453    1339      2343     416    1851    1173     242
Far North Coast  2023               1278                 245            33     101     596       927     153     856     417      97
         Hunter  2023               1965                 673           107     206    1011      1627     301    1210     783     344
      Illawarra  2023               1310                 200            34     342     601       909     108     704     559     139
     Inner West  2023               1946                 396             9    1833     908      1434     196    1107     798     241
Mid North Coast  2023               1686                 547         

In [36]:
# =============================================================================
# STEP 4 — Save
# =============================================================================

summary.to_csv(OUT_HOME, index=False)
print(f'Saved: {OUT_HOME}')
print(f'Shape: {summary.shape}')

Saved: ../../data/clean/homecare_admissions_by_acpr.csv
Shape: (454, 27)


---

## Part 2 — Home Care Users (service_users_CURF)

### Input
- `data/raw/service_users_CURF/` — CURF snapshots 2018–2024

### Output
- `data/clean/home_care_users_by_acpr.csv` — one row per ACPR × year

### Columns
- `year`, `state`, `acpr_code`, `acpr_name`
- `hcp_l1`, `hcp_l2`, `hcp_l3`, `hcp_l4`
- `n_age_<group>`
- `total_users`

### Note
- 2018–2019 file is GEN data (pre-aggregated, has `count` column); 2020–2024 are individual-level CURFs
- Keeps Home care only

In [37]:
RAW_CURF  = '../../data/raw/service_users_CURF'
OUT_USERS = '../../data/clean/home_care_users_by_acpr.csv'

WANTED_CURF = {
    'ACPR_code', 'ACPR_CODE', 'ACPR_CODE_2018',
    'ACPR_name', 'ACPR_NAME', 'ACPR_NAME_2018',
    'State', 'STATE',
    'Year', 'YEAR',
    'Care_type', 'CARE_TYPE',
    'Home_care_level', 'HOME_CARE_LEVEL',
    'Age_group', 'AGE_GROUP', 'AGE_GROUP_5',
    'Sex', 'SEX',
    'Indigenous_status', 'INDIGENOUS_STATUS',
    'Country_of_birth', 'COUNTRY_OF_BIRTH',
    'count',
}

COL_MAP_CURF = {
    'ACPR_code': 'acpr_code', 'ACPR_CODE': 'acpr_code', 'ACPR_CODE_2018': 'acpr_code',
    'ACPR_name': 'acpr_name', 'ACPR_NAME': 'acpr_name', 'ACPR_NAME_2018': 'acpr_name',
    'State': 'state', 'STATE': 'state',
    'Year': 'year', 'YEAR': 'year',
    'Care_type': 'care_type', 'CARE_TYPE': 'care_type',
    'Home_care_level': 'hcp_level', 'HOME_CARE_LEVEL': 'hcp_level',
    'Age_group': 'age_group', 'AGE_GROUP': 'age_group', 'AGE_GROUP_5': 'age_group',
    'Sex': 'sex', 'SEX': 'sex',
    'Indigenous_status': 'indigenous_status', 'INDIGENOUS_STATUS': 'indigenous_status',
    'Country_of_birth': 'country_of_birth', 'COUNTRY_OF_BIRTH': 'country_of_birth',
}

In [38]:
# =============================================================================
# STEP 5 — Process home care users (service_users_CURF)
# =============================================================================

def aggregate_curf_file(df, fname):
    # 1. Slim and rename
    keep = [c for c in df.columns if c in WANTED_CURF]
    df = df[keep].copy()
    df = df.rename(columns={k: v for k, v in COL_MAP_CURF.items() if k in df.columns})

    if 'acpr_code' not in df.columns:
        print(f'  SKIP {fname} — no acpr_code')
        return None

    # 2. Clean acpr_code and year
    df['acpr_code'] = df['acpr_code'].astype(str).str.strip()
    df = df[df['acpr_code'].str.len() > 0].dropna(subset=['acpr_code'])
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df = df.dropna(subset=['year'])
    df['year'] = df['year'].astype(int)

    # 3. Keep Home care only
    if 'care_type' in df.columns:
        df = df[df['care_type'].astype(str).str.strip().str.lower() == 'home care']
    if df.empty:
        print(f'  SKIP {fname} — no home care rows')
        return None

    # 4. Row weight: GEN files have a 'count' column; individual CURFs default to 1
    if 'count' in df.columns:
        df['_weight'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)
    else:
        df['_weight'] = 1

    # 5. Binary flags (all multiplied by weight)
    sex = df['sex'].astype(str).str.strip().str.lower() if 'sex' in df.columns else pd.Series('', index=df.index)
    df['_male']   = sex.isin(['male', 'males', '1']).astype(int) * df['_weight']
    df['_female'] = sex.isin(['female', 'females', '2']).astype(int) * df['_weight']

    # Indigenous: string files use 'Indigenous'/'Non-Indigenous';
    # 2018-2019 GEN uses numeric codes 1=Aboriginal, 2=TSI, 3=Both, 4=Non-Indigenous
    ind = df['indigenous_status'].astype(str).str.strip().str.lower() if 'indigenous_status' in df.columns else pd.Series('', index=df.index)
    df['_indigenous'] = ind.isin(['indigenous', '1', '2', '3']).astype(int) * df['_weight']

    cob = df['country_of_birth'].astype(str).str.strip().str.lower() if 'country_of_birth' in df.columns else pd.Series('', index=df.index)
    df['_nesb']    = cob.str.contains('non-english', na=False).astype(int) * df['_weight']
    df['_english'] = (cob.str.contains('english', na=False) & ~cob.str.contains('non-english', na=False)).astype(int) * df['_weight']

    # 6. HCP level flags × weight
    if 'hcp_level' in df.columns:
        hcp_norm = df['hcp_level'].astype(str).str.strip().str.lower().map(HCP_MAP)
        for col in HCP_COLS:
            df[f'_{col}'] = (hcp_norm == col).astype(int) * df['_weight']
    else:
        for col in HCP_COLS:
            df[f'_{col}'] = 0

    # 7. Age bracket flags × weight
    if 'age_group' in df.columns:
        age_norm = df['age_group'].astype(str).str.strip().map(AGE_MAP)
        for col in AGE_COLS:
            df[f'_{col}'] = (age_norm == col).astype(int) * df['_weight']
    else:
        for col in AGE_COLS:
            df[f'_{col}'] = 0

    # 8. Aggregate per ACPR × year
    gcols = [c for c in GROUP_COLS if c in df.columns]
    agg_map = {
        'total_users':        ('_weight',     'sum'),
        'n_male':             ('_male',        'sum'),
        'n_female':           ('_female',      'sum'),
        'n_indigenous':       ('_indigenous',  'sum'),
        'n_nesb':             ('_nesb',        'sum'),
        'n_english_speaking': ('_english',     'sum'),
    }
    for col in HCP_COLS:
        agg_map[col] = (f'_{col}', 'sum')
    for col in AGE_COLS:
        agg_map[col] = (f'_{col}', 'sum')

    return df.groupby(gcols).agg(**agg_map).reset_index()


results_curf = []
for fname in sorted(os.listdir(RAW_CURF)):
    if fname.startswith('~'): continue
    if not (fname.endswith('.xlsx') or fname.endswith('.csv')): continue

    path = f'{RAW_CURF}/{fname}'
    try:
        df  = load_file(path)
        agg = aggregate_curf_file(df, fname)
        if agg is not None:
            results_curf.append(agg)
            print(f'OK  {fname}: {df.shape} -> {agg.shape}  years={sorted(agg["year"].unique())}')
        del df
    except Exception as e:
        print(f'ERR {fname}: {e}')

users = pd.concat(results_curf, ignore_index=True)
print(f'\nCombined: {users.shape}')
print('Years:', sorted(users['year'].unique()))
print('ACPRs:', users['acpr_code'].nunique())

OK  CURF_People-using_aged_care-2024.xlsx: (478720, 11) -> (72, 26)  years=[np.int64(2024)]
OK  People-using-aged-care-serrvices-30-June-2021.csv: (370848, 12) -> (72, 26)  years=[np.int64(2021)]
OK  People-using-aged-care-services-30-June-2020.xlsx: (335889, 12) -> (72, 26)  years=[np.int64(2020)]
OK  People-using-aged-care-services-30-June-2022.csv: (403950, 11) -> (72, 26)  years=[np.int64(2022)]
OK  People-using-aged-care-services-30-June-2023.xlsx: (455082, 11) -> (72, 26)  years=[np.int64(2023)]
OK  People_2018to2019_GENdata_version_2.csv: (484253, 26) -> (72, 26)  years=[np.int64(2019)]

Combined: (432, 26)
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
ACPRs: 144


In [39]:
# =============================================================================
# STEP 6 — Quick sanity check
# =============================================================================

print(users[['acpr_name', 'year', 'total_users',
             'n_male', 'n_female', 'n_indigenous', 'n_nesb',
             'hcp_l1', 'hcp_l2', 'hcp_l3', 'hcp_l4']].head(10).to_string(index=False))
print()
print('Age cols sample:')
print(users[['acpr_name', 'year'] + AGE_COLS].head(5).to_string(index=False))

      acpr_name  year  total_users  n_male  n_female  n_indigenous  n_nesb  hcp_l1  hcp_l2  hcp_l3  hcp_l4
  Central Coast  2024         5142    1695      3446           221     527     373    2332    1668     769
   Central West  2024         5572    1810      3762           841     633     569    2825    1664     514
Far North Coast  2024         3578    1225      2353           129     211     293    1735    1054     496
         Hunter  2024         6360    2058      4302           316     537     651    2751    1749    1209
      Illawarra  2024         4080    1315      2764           194     943     269    1708    1412     691
     Inner West  2024         6155    2011      4142            24    5010     367    2791    2078     919
Mid North Coast  2024         5948    2175      3771           389     334     465    2414    2025    1044
         Nepean  2024         1960     629      1331            41     574     112     899     667     282
    New England  2024         1266   

In [40]:
# =============================================================================
# STEP 6b — Sanity check: compare CURF totals vs GEN snapshot (2023 & 2024)
#
# Snapshot = point-in-time count on 30 June (people IN care that day)
# CURF     = all users at any point during the year → expect CURF >= snapshot
# Numbers should be in the same ballpark (same order of magnitude, levels proportional)
# =============================================================================

SNAP_DIR = '../../data/raw/service_users_snapshot_SA3'
SNAP_FILES = {
    2023: f'{SNAP_DIR}/GEN-data-People-using-aged-care-by-region-30-June-2023-2-home-care-(recipient-location).xlsx',
    2024: f'{SNAP_DIR}/GEN-data-People-using-aged-care-by-region-30-June-2024-2-home-care-(recipient-location).xlsx',
}

def load_snapshot_acpr(path):
    raw = pd.read_excel(path, sheet_name='Table 2.3 (ACPR)', header=None, engine='calamine')
    # Find the header row (contains 'Level 1')
    header_row = 0
    for i, row in raw.iterrows():
        if any('level 1' in str(v).lower() for v in row.values):
            header_row = i
            break
    df = pd.read_excel(path, sheet_name='Table 2.3 (ACPR)', header=header_row, engine='calamine')
    df.columns = [str(c).strip() for c in df.columns]
    # Keep only data rows (first col is a numeric ACPR code)
    code_col = df.columns[0]
    df = df[pd.to_numeric(df[code_col], errors='coerce').notna()].copy()
    return df

rows = []
for yr in [2023, 2024]:
    snap = load_snapshot_acpr(SNAP_FILES[yr])
    # Sum nationally — find level/total columns by name
    level_cols = [c for c in snap.columns if 'level' in c.lower()]
    total_col  = [c for c in snap.columns if c.lower() == 'total']

    snap_l = snap[level_cols].sum().values.tolist()   # [L1, L2, L3, L4]
    snap_t = int(snap[total_col[0]].sum()) if total_col else sum(snap_l)

    curf_yr = users[users['year'] == yr]
    curf_l  = [int(curf_yr[c].sum()) for c in HCP_COLS]
    curf_t  = int(curf_yr['total_users'].sum())

    rows.append({
        'year':            yr,
        'snap_l1':         int(snap_l[0]), 'curf_l1': curf_l[0],
        'snap_l2':         int(snap_l[1]), 'curf_l2': curf_l[1],
        'snap_l3':         int(snap_l[2]), 'curf_l3': curf_l[2],
        'snap_l4':         int(snap_l[3]), 'curf_l4': curf_l[3],
        'snap_total':      snap_t,         'curf_total': curf_t,
    })

comp = pd.DataFrame(rows).set_index('year')
print('Snapshot (30 Jun point-in-time) vs CURF (full-year users)\n')
print(comp.to_string())
print('\nCURF / Snapshot ratio (expect > 1, typically 1.1–1.5x):')
for col in ['l1', 'l2', 'l3', 'l4', 'total']:
    for yr in [2023, 2024]:
        s = comp.loc[yr, f'snap_{col}']
        c = comp.loc[yr, f'curf_{col}']
        ratio = c / s if s > 0 else float('nan')
        print(f'  {yr} {col}: {ratio:.2f}')

Snapshot (30 Jun point-in-time) vs CURF (full-year users)

      snap_l1  curf_l1  snap_l2  curf_l2  snap_l3  curf_l3  snap_l4  curf_l4  snap_total  curf_total
year                                                                                                
2023    13425    13439   103294   103676    87311    87447    53657    53812      257687      258374
2024    14773    14792   111661   111996    90762    90908    57624    57790      274820      275486

CURF / Snapshot ratio (expect > 1, typically 1.1–1.5x):
  2023 l1: 1.00
  2024 l1: 1.00
  2023 l2: 1.00
  2024 l2: 1.00
  2023 l3: 1.00
  2024 l3: 1.00
  2023 l4: 1.00
  2024 l4: 1.00
  2023 total: 1.00
  2024 total: 1.00


In [41]:
# =============================================================================
# STEP 7 — Save
# =============================================================================

users.to_csv(OUT_USERS, index=False)
print(f'Saved: {OUT_USERS}')
print(f'Shape: {users.shape}')

Saved: ../../data/clean/home_care_users_by_acpr.csv
Shape: (432, 26)
